# H&M M1과 M3+M4-B의 후보별 오류 진단

기존 H&M 개발구간 2020-09-02~08, seed43, 300epoch checkpoint만 읽습니다. 새 학습·epoch 선택·마지막 주 test·holdout 없음. **GPU + 고용량 RAM 런타임**에서 실행하세요.

필수 Drive 자료는 `results_v3_hm_clv_m2_training_budget_seed43_v1`의 M1 결과/checkpoint와 `results_v3_hm_clv_m5_m3_m4_split_nv_hm2y_s43_v1`의 M3+M4-B 결과/checkpoint, 원래 H&M 입력 데이터입니다. 하나라도 없거나 입력·출처가 다르면 재학습하지 않고 중단합니다.

전체 지표를 기존 기록과 재현한 뒤 정답 Top100 순위와 Top50 추천을 저장합니다. 상품 구매고객 수는 전체 TRAIN 그래프에서, 카테고리·가격대 문맥은 평가 고객의 TRAIN 이력에서만 계산합니다. 결과는 Dunnhumby와 방향을 교차 확인하는 기술통계이며 인과·유의성·CLV 귀속·새 모델 성공을 주장하지 않습니다.

로컬 합성 검사는 통과했지만 실제 Drive 전체 실행과 메모리·시간은 아직 검증되지 않았습니다. 아래 셀을 위에서부터 실행하고 ZIP을 전달하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json
SOURCE_COMMIT = '9140b889a8bdd5bdae2d6e38219765f6b7763f2e'
repo = Path('/content/clv-hm-m1-m5-diagnostic-' + SOURCE_COMMIT[:12])
if not repo.exists():
    subprocess.run(['git','clone','-q','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','-q','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout','-q','--detach',SOURCE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip() == SOURCE_COMMIT
sys.path.insert(0,str(repo))
root = Path('/content/drive/MyDrive/논문/data')
required = [root/'results_v3_hm_clv_m2_training_budget_seed43_v1',
            root/'results_v3_hm_clv_m5_m3_m4_split_nv_hm2y_s43_v1']
missing = [str(path) for path in required if not path.is_dir()]
if missing:
    raise FileNotFoundError({'missing_existing_result_folders': missing, 'new_training': False})
print(json.dumps({'source_commit': SOURCE_COMMIT, 'existing_folders': [str(p) for p in required],
                  'new_training': False, 'split': '2020-09-02~08', 'seed': 43, 'epoch': 300},
                 ensure_ascii=False, indent=2))

## 기존 체크포인트 읽기·재현·후보 진단

전체 카탈로그를 배치 단위로 GPU 점수화합니다. 학습쌍은 후보에서 제외합니다. 기존 300epoch 전체 지표와 재현 차이가 있으면 `readback.csv`를 남기고 중단합니다. 실행 중 런타임이 끊기면 학습 상태에는 영향이 없지만 진단 계산은 처음부터 다시 실행합니다.

In [ ]:
import torch
assert torch.cuda.is_available(), '런타임 유형을 GPU로 바꾸세요.'
import clv_hm_m1_m5_error_diagnostic as diagnostic
paths = diagnostic.run(root)
print(json.dumps(paths, ensure_ascii=False, indent=2))

## 결과 확인·원본 다운로드

단일 개발 seed이므로 H&M 내부 안정성이나 유의성을 판단하지 않습니다. `item_buyers`는 진단용 대중성 변수이지 CLV 제안기법 입력으로 확정된 것이 아닙니다. `feature_summary`는 후보쌍 평균과 고객별 평균을 함께 저장하고, `paired_user_feature_differences`는 같은 고객의 공통 누락과 각 모델 오추천 차이를 저장합니다.

In [ ]:
import pandas as pd
from IPython.display import display
summary = pd.read_csv(paths['summary'])
display(summary[summary.grouping.isin(['all','segment'])])
feature_summary = pd.read_csv(paths['feature_summary'])
display(feature_summary[(feature_summary.k == 10) & feature_summary.segment.isin(['all','고CLV'])])
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive_path = Path('/content/clv_hm_m1_m5_error_diagnostic_seed43.zip')
with ZipFile(archive_path, 'w', ZIP_DEFLATED) as archive:
    for path in paths.values():
        archive.write(path, arcname=Path(path).name)
files.download(str(archive_path))